# First prototype for HyperBubble Resolution-Oriented GNN

In [93]:
from pathlib import Path
from typing import Any, Dict, List, Optional
import json
import torch
from torch.utils.data import Dataset
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch.utils.data import Subset
import torch.nn as nn
import torch.optim as opt
import torch.nn.functional as F
from torch_geometric.nn import GraphNorm

In [94]:
NUC2ID = { 'A':1, 'C':2, 'G':3, 'T':4, 'N':5 }
def seq_to_tokens(seq: str) -> torch.LongTensor:
    return torch.tensor([NUC2ID.get(c, 5) for c in seq], dtype=torch.long)

def _safe_get(d: Dict[str, Any], key: str, default):
    v = d.get(key, default)
    return default if v is None else v

ID2NUC = {v:k for k,v in NUC2ID.items()}

def tokens_to_seq(tokens: torch.LongTensor) -> str:
    return "".join(ID2NUC.get(int(t), "N") for t in tokens.tolist())

In [95]:
import json
from pathlib import Path
from typing import List, Dict, Any, Optional
import torch
from torch.utils.data import Dataset
from torch_geometric.data import Data

class HyperbubbleDataset(Dataset):
    """
    Minimal GNN-ready dataset from your JSONL:
      - Node features:
          seq_tokens : [N, K] Long
          x_cov      : [N, 1] Float (KC coverage; 0 if unknown)
      - Graph:
          edge_index : [2, E] Long
          edge_attr  : [E, 4] Float (len_nodes, len_bp, cov_min, cov_mean)
          start_idx, end_idx : Long
      - Labels:
          y_edge_mask    : [E] Long (1 on labeled path edges, else 0)
          label_path_idx : Long (-1 if none)
    """
    def __init__(self, files: List[str]):
        self.files = [Path(p) for p in files]
        self.records: List[Dict[str, Any]] = []
        for fp in self.files:
            with fp.open("r") as fh:
                for line in fh:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        self.records.append(json.loads(line))
                    except json.JSONDecodeError:
                        continue

    def __len__(self) -> int:
        return len(self.records)

    def _build_graph(self, rec: Dict[str, Any]) -> Data:
        # --- nodes (map seq -> idx), carry best-known coverage ---
        node_seqs: Dict[str, int] = {}
        node_covs: List[float] = []

        def ensure_node(seq: str, cov: Optional[float] = None) -> int:
            if seq not in node_seqs:
                node_seqs[seq] = len(node_seqs)
                node_covs.append(float(cov) if cov is not None else 0.0)
            else:
                i = node_seqs[seq]
                if cov is not None and node_covs[i] == 0.0:
                    node_covs[i] = float(cov)
            return node_seqs[seq]

        # endpoints
        start_seq = rec["start_seq"]
        end_seq   = rec["end_seq"]

        # core nodes
        for n in _safe_get(rec, "nodes", []):
            ensure_node(n["seq"], n.get("cov", 0))

        # 1-hop context
        for n in _safe_get(rec, "upstream_nodes", []):
            ensure_node(n["seq"], n.get("cov", 0))
        for n in _safe_get(rec, "downstream_nodes", []):
            ensure_node(n["seq"], n.get("cov", 0))

        # ---- Strategy-A selected interior nodes (if present) ----
        for n in _safe_get(rec, "inside_nodes", []):
            if isinstance(n, dict) and "seq" in n:
                ensure_node(n["seq"], n.get("cov", 0))

        # make sure endpoints are present
        ensure_node(start_seq)
        ensure_node(end_seq)

        # --- edges + attributes ---
        edge_src, edge_dst, edge_attr = [], [], []
        edge_list = _safe_get(rec, "all_edges", None)
        if not edge_list:
            edge_list = _safe_get(rec, "edges", [])
        for e in edge_list:
            u = ensure_node(e["source_seq"])
            v = ensure_node(e["target_seq"])
            edge_src.append(u)
            edge_dst.append(v)
            edge_attr.append([
                float(e.get("len_nodes", 0)),
                float(e.get("len_bp", 0)),
                float(e.get("cov_min", 0)),
                float(e.get("cov_mean", 0.0))
            ])

        # --- node features ---
        node_order = [None] * len(node_seqs)
        for s, idx in node_seqs.items():
            node_order[idx] = s

        # keep tokens; embedding happens in GNN
        seq_tokens = torch.stack([seq_to_tokens(s) for s in node_order], dim=0)  # [N, K] Long
        x_cov = torch.tensor(node_covs, dtype=torch.float32).unsqueeze(1)        # [N, 1] Float

        start_idx = torch.tensor(node_seqs[start_seq], dtype=torch.long)
        end_idx   = torch.tensor(node_seqs[end_seq], dtype=torch.long)

        edge_index = torch.tensor([edge_src, edge_dst], dtype=torch.long)
        edge_attr  = (torch.tensor(edge_attr, dtype=torch.float32)
                      if edge_attr else torch.zeros((0,5), dtype=torch.float32))

        # --- labels from label_path ---
        num_edges = edge_index.size(1)
        y_edge_mask = torch.zeros(num_edges, dtype=torch.long)
        label_path_idx = -1
        paths_list = _safe_get(rec, "paths", [])
        lp = rec.get("label_path")
        if lp is not None and 0 <= lp < len(paths_list) and num_edges > 0:
            label_path_idx = int(lp)
            y_edge_mask[torch.tensor(paths_list[label_path_idx], dtype=torch.long)] = 1

        data = Data(
            seq_tokens=seq_tokens,                 # [N, K] Long (to be embedded in model)
            x_cov=x_cov,                           # [N, 1] Float
            edge_index=edge_index,                 # [2, E] Long
            edge_attr=edge_attr,                   # [E, 5] Float
            start_idx=start_idx,                   # Long
            end_idx=end_idx,                       # Long
            num_nodes=seq_tokens.size(0),
            y_edge_mask=y_edge_mask,               # [E] Long
            label_path_idx=torch.tensor(label_path_idx, dtype=torch.long),
        )
        if "bubble_id" in rec:
            data.bubble_id = torch.tensor(rec["bubble_id"], dtype=torch.long)
        if "k" in rec:
            data.k = torch.tensor(rec["k"], dtype=torch.long)
        return data

    def __getitem__(self, idx: int) -> Data:
        return self._build_graph(self.records[idx])
        
# Keep only labeled samples
def labeled_subset(ds):
    labeled_indices = []
    for i in range(len(ds)):
        d = ds[i]
        lp = int(d.label_path_idx.item()) if hasattr(d, "label_path_idx") else -1
        if lp >= 0:
            labeled_indices.append(i)
    return Subset(ds, labeled_indices)

In [96]:
device = torch.device('cpu')
if torch.cuda.is_available():
    device = torch.device('cuda')

    
class EdgeMPNNLayer(nn.Module):
    """
    msg(u->v) = W_u h_u + MLP(edge_attr)
    h'_v = W_self h_v + mean_{u->v} msg(u->v)
    """
    def __init__(self, in_dim: int, out_dim: int, edge_feat_dim: int, edge_hidden: int = None):
        super().__init__()
        if edge_hidden is None:
            edge_hidden = out_dim

        self.lin_u = nn.Linear(in_dim, out_dim)
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.edge_mlp = nn.Sequential(
            nn.Linear(edge_feat_dim, edge_hidden),
            nn.ReLU(),
            nn.Linear(edge_hidden, out_dim),
        )

    def forward(self, h: torch.Tensor, edge_index: torch.Tensor, edge_attr: torch.Tensor) -> torch.Tensor:
        n = h.size(0)
        out = self.lin_self(h)

        if edge_index.numel() == 0:
            return out

        src, dst = edge_index[0], edge_index[1]          # [E], [E]
        msg = self.lin_u(h[src]) + self.edge_mlp(edge_attr)  # [E, out_dim]

        agg = h.new_zeros((n, msg.size(1)))
        agg.index_add_(0, dst, msg)

        deg = h.new_zeros((n, 1))
        deg.index_add_(0, dst, h.new_ones((dst.numel(), 1)))
        out = out + agg / deg.clamp(min=1.0)  # mean

        return out


    
class HyperbubbleGNN(nn.Module):
    def __init__(
        self,
        vocab_size=6,
        embed_dim=32,
        gcn_hidden=64,
        edge_hidden=64,
        edge_feat_dim=4,
        use_lstm=False,
        dropout=0.2,
        edge_dropout=0.1,
    ):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.use_lstm = use_lstm
        self.edge_feat_dim = edge_feat_dim
        self.dropout = float(dropout)
        self.edge_dropout = float(edge_dropout)

        if use_lstm:
            self.lstm = nn.LSTM(embed_dim, gcn_hidden // 2, batch_first=True, bidirectional=False)
            self.node_in = gcn_hidden + 1
        else:
            self.node_in = embed_dim + 1

        self.mp1 = EdgeMPNNLayer(self.node_in, gcn_hidden, edge_feat_dim, edge_hidden)
        self.gn1 = GraphNorm(gcn_hidden)

        self.mp2 = EdgeMPNNLayer(gcn_hidden, gcn_hidden, edge_feat_dim, edge_hidden)
        self.gn2 = GraphNorm(gcn_hidden)

        self.edge_mlp = nn.Sequential(
            nn.Linear(2 * gcn_hidden + edge_feat_dim, edge_hidden),
            nn.ReLU(),
            nn.Dropout(self.dropout),
            nn.Linear(edge_hidden, 1),
        )

    def encode_nodes(self, seq_tokens, x_cov):
        E = self.embed(seq_tokens)
        mask = (seq_tokens != 0).float().unsqueeze(-1)

        if self.use_lstm:
            lengths = mask.squeeze(-1).sum(dim=1).clamp(min=1).long()
            H0, _ = self.lstm(E)
            Hseq = (H0 * mask).sum(dim=1) / lengths.unsqueeze(-1).float()
        else:
            lengths = mask.squeeze(-1).sum(dim=1).clamp(min=1)
            Hseq = (E * mask).sum(dim=1) / lengths.unsqueeze(-1)

        # log1p on coverage often helps a lot
        x_cov = torch.log1p(x_cov.clamp(min=0))
        return torch.cat([Hseq, x_cov], dim=1)

    def _fix_edge_attr(self, edge_attr, E, device, dtype):
        if edge_attr is None or edge_attr.numel() == 0:
            EA = torch.zeros((E, self.edge_feat_dim), device=device, dtype=dtype)
        else:
            EA = edge_attr
            if EA.dim() != 2:
                EA = EA.view(E, -1)
            if EA.size(1) > self.edge_feat_dim:
                EA = EA[:, :self.edge_feat_dim]
            elif EA.size(1) < self.edge_feat_dim:
                pad = torch.zeros((E, self.edge_feat_dim - EA.size(1)), device=EA.device, dtype=EA.dtype)
                EA = torch.cat([EA, pad], dim=1)
            EA = EA.to(device=device, dtype=dtype)

        # Minimal scaling:
        # lengths: log1p
        EA[:, 0:2] = torch.log1p(EA[:, 0:2].clamp(min=0))
        # cov_min/cov_mean: log1p
        EA[:, 2:4] = torch.log1p(EA[:, 2:4].clamp(min=0))

        if self.training and self.edge_dropout > 0:
            EA = F.dropout(EA, p=self.edge_dropout, training=True)

        return EA

    def forward(self, data):
        device = data.seq_tokens.device
        N = int(data.num_nodes)
        E = int(data.edge_index.size(1))
        batch = data.batch if hasattr(data, "batch") else torch.zeros(N, dtype=torch.long, device=device)

        X0 = self.encode_nodes(data.seq_tokens, data.x_cov)

        if E == 0:
            return torch.empty((0,), device=device)

        EA = self._fix_edge_attr(getattr(data, "edge_attr", None), E, device, X0.dtype)

        H = self.mp1(X0, data.edge_index, EA)
        H = self.gn1(H, batch)
        H = F.relu(H)
        H = F.dropout(H, p=self.dropout, training=self.training)

        H = self.mp2(H, data.edge_index, EA)
        H = self.gn2(H, batch)
        H = F.relu(H)
        H = F.dropout(H, p=self.dropout, training=self.training)

        u, v = data.edge_index
        feats = torch.cat([H[u], H[v], EA], dim=1)
        return self.edge_mlp(feats).squeeze(-1)

In [97]:
def _num_graphs_in_batch(node_batch: torch.Tensor) -> int:
    return int(node_batch.max().item()) + 1 if node_batch.numel() else 0

def _edge_batch_from_nodes(node_batch: torch.Tensor, edge_src: torch.Tensor) -> torch.Tensor:
    return node_batch[edge_src] if edge_src.numel() else edge_src.new_zeros((0,), dtype=torch.long)

def _collect_decisions_by_labeled_sources(data, g: int, u: torch.Tensor, edge_batch: torch.Tensor):
    y = data.y_edge_mask
    lab_eidx = (y == 1).nonzero(as_tuple=False).view(-1)
    lab_eidx = lab_eidx[edge_batch[lab_eidx] == g]
    if lab_eidx.numel() == 0:
        return []

    labeled_sources = torch.unique(u[lab_eidx])
    decisions = []
    for s in labeled_sources.tolist():
        mask = (edge_batch == g) & (u == s)
        cand_idx = mask.nonzero(as_tuple=False).view(-1)
        if cand_idx.numel() < 2:
            continue
        y_here = y[cand_idx]
        gold_pos = (y_here == 1).nonzero(as_tuple=False).view(-1)
        if gold_pos.numel() == 1:
            decisions.append((cand_idx, gold_pos.view(1)))
    return decisions

def train_one_epoch_choice_all(model, loader, optim, device):
    model.train()

    total_loss_sum = 0.0
    total_decisions = 0
    total_correct = 0

    for data in loader:
        data = data.to(device)
        logits = model(data)
        if logits.numel() == 0:
            continue

        u = data.edge_index[0]
        node_batch = data.batch
        edge_batch = _edge_batch_from_nodes(node_batch, u)
        num_graphs = _num_graphs_in_batch(node_batch)

        batch_loss_sum = 0.0
        batch_decisions = 0
        batch_correct = 0

        for g in range(num_graphs):
            decisions = _collect_decisions_by_labeled_sources(data, g, u, edge_batch)
            for cand_idx, gold_pos in decisions:
                ce = F.cross_entropy(logits[cand_idx].unsqueeze(0), gold_pos.view(1))
                batch_loss_sum += ce
                batch_decisions += 1

                pred = logits[cand_idx].argmax().item()
                gold = gold_pos.item()
                batch_correct += int(pred == gold)

        if batch_decisions == 0:
            continue

        batch_loss = batch_loss_sum / batch_decisions
        optim.zero_grad()
        batch_loss.backward()
        optim.step()

        total_loss_sum += float(batch_loss.item()) * batch_decisions
        total_decisions += batch_decisions
        total_correct += batch_correct

    train_loss = total_loss_sum / max(total_decisions, 1)
    train_dec = total_correct / max(total_decisions, 1)
    return {"loss": train_loss, "dec_acc": train_dec, "decisions": total_decisions}

@torch.no_grad()
def eval_choice_all(model, loader, device, final_eval=False):
    model.eval()

    total_loss_sum = 0.0
    total_correct = 0
    total_decisions = 0

    total_bubbles = 0
    correct_bubbles = 0

    for data in loader:
        data = data.to(device)
        logits = model(data)
        if logits.numel() == 0:
            continue

        u = data.edge_index[0]
        node_batch = data.batch
        edge_batch = _edge_batch_from_nodes(node_batch, u)
        num_graphs = _num_graphs_in_batch(node_batch)

        for g in range(num_graphs):
            decisions = _collect_decisions_by_labeled_sources(data, g, u, edge_batch)
            if not decisions:
                continue

            total_bubbles += 1
            all_correct = True

            for cand_idx, gold_pos in decisions:
                ce = F.cross_entropy(logits[cand_idx].unsqueeze(0), gold_pos.view(1))
                total_loss_sum += float(ce.item())
                total_decisions += 1

                pred = logits[cand_idx].argmax().item()
                ok = int(pred == gold_pos.item())
                total_correct += ok
                all_correct &= bool(ok)

            correct_bubbles += int(all_correct)

    val_loss = total_loss_sum / max(total_decisions, 1)
    val_dec  = total_correct / max(total_decisions, 1)
    bub_acc  = correct_bubbles / max(total_bubbles, 1)

    if final_eval:
        print(f"[test] test_loss={val_loss:.4f}  test_dec={val_dec:.4f}  test_bub={bub_acc:.4f} "
              f"(decisions={total_decisions} bubbles={total_bubbles})")

    return val_loss, val_dec


def print_run_header(tag: str, fold: int, test_name: str, n_train: int, n_val: int, n_test: int):
    print(f"[{tag}] fold={fold} test={test_name} (train={n_train} val={n_val} test={n_test})")

def print_epoch_line(ep: int, train_loss: float, train_dec: float, val_loss: float, val_dec: float):
    print(
        f"[ep {ep:03d}] "
        f"train_loss={train_loss:.4f}  train_dec={train_dec:.4f}  "
        f"val_loss={val_loss:.4f}  val_dec={val_dec:.4f}"
    )

def print_test_line(test_loss: float, test_dec: float, test_bub: float, decisions: int, bubbles: int):
    print(
        f"[test] "
        f"test_loss={test_loss:.4f}  test_dec={test_dec:.4f}  test_bub={test_bub:.4f}  "
        f"(decisions={decisions} bubbles={bubbles})"
    )


In [106]:
TRAIN_PATHS = [
    "lower_cov_data/Salmonella_dataset_cov20_ratio_lt_3.5_all_edges.jsonl",
            "lower_cov_data/Klebsiella_dataset_cov20_ratio_lt_3.5_all_edges.jsonl",
    

]
TEST_PATHS = [
    "lower_cov_data/Ecoli_dataset_cov20_ratio_lt_3.5_all_edges.jsonl",
]

# sanity check
for p in TRAIN_PATHS + TEST_PATHS:
    if not Path(p).is_file():
        raise FileNotFoundError(f"Missing: {p}")

train_full = HyperbubbleDataset(TRAIN_PATHS)
test_full  = HyperbubbleDataset(TEST_PATHS)

# labeled only
train_labeled = labeled_subset(train_full)
test_dataset  = labeled_subset(test_full)

val_fraction = 0.1
n_total = len(train_labeled)
n_val   = int(n_total * val_fraction)
n_train = n_total - n_val

train_idx, val_idx = torch.utils.data.random_split(
    range(n_total),
    lengths=[n_train, n_val],
    generator=torch.Generator().manual_seed(1337)
)

train_dataset = Subset(train_labeled, train_idx.indices)
val_dataset   = Subset(train_labeled, val_idx.indices)

print(f"Train labeled: {len(train_dataset)}")
print(f"Val labeled:   {len(val_dataset)}")
print(f"Test labeled:  {len(test_dataset)}")

# Loaders
BATCH_SIZE = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

Train labeled: 702
Val labeled:   78
Test labeled:  344


In [99]:

# # -------- model / optim --------
# model = HyperbubbleGNN(
#     vocab_size=6,
#     embed_dim=32,
#     gcn_hidden=128,
#     edge_hidden=64,
#     edge_feat_dim=4,
#     use_lstm=False,
# ).to(device)

# total_params = sum(p.numel() for p in model.parameters())
# print(f"Total params: {total_params}")

# optim = opt.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-3)
# # -------- train / eval loop with best checkpoint --------
# best_ckpt = {"epoch": 0, "dec_acc": -1.0, "bub_acc": -1.0, "state_dict": None}

# def _is_better(bub_acc, dec_acc, best):
#     if dec_acc > best["dec_acc"]:
#         return True
#     if dec_acc == best["dec_acc"] and bub_acc > best["bub_acc"]:
#         return True
#     return False

# EPOCHS = 500
# for epoch in range(1, EPOCHS + 1):
#     tr = train_one_epoch_choice_all(model, train_loader, optim, device)
#     tr_loss = tr["loss"]
#     tr_dec  = tr["dec_acc"]

#     val_loss, val_dec = eval_choice_all(model, val_loader, device)

#     print(f"[ep {epoch:03d}] "
#           f"train_loss={tr_loss:.4f}  train_dec={tr_dec:.4f}  "
#           f"val_loss={val_loss:.4f}  val_dec={val_dec:.4f}")

#     # checkpoint logic
#     dec_acc, bub_acc = val_dec, 0.0
#     if _is_better(bub_acc, dec_acc, best_ckpt):
#         best_ckpt["epoch"] = epoch
#         best_ckpt["dec_acc"] = dec_acc
#         best_ckpt["bub_acc"] = bub_acc
#         best_ckpt["state_dict"] = {
#             k: v.detach().cpu() for k, v in model.state_dict().items()
#         }
#         torch.save(best_ckpt, "best_hyperbubble.ckpt")


Total params: 76737
[ep 001] train_loss=0.7025  train_dec=0.5013  val_loss=0.6923  val_dec=0.5833
[ep 002] train_loss=0.7009  train_dec=0.4973  val_loss=0.6950  val_dec=0.5714
[ep 003] train_loss=0.6925  train_dec=0.5539  val_loss=0.6942  val_dec=0.5595
[ep 004] train_loss=0.7001  train_dec=0.5013  val_loss=0.6932  val_dec=0.5476
[ep 005] train_loss=0.6949  train_dec=0.5404  val_loss=0.6931  val_dec=0.5595
[ep 006] train_loss=0.6963  train_dec=0.5256  val_loss=0.6940  val_dec=0.5595
[ep 007] train_loss=0.6921  train_dec=0.5350  val_loss=0.6932  val_dec=0.5595
[ep 008] train_loss=0.6948  train_dec=0.5148  val_loss=0.6927  val_dec=0.5595
[ep 009] train_loss=0.6938  train_dec=0.5350  val_loss=0.6921  val_dec=0.5595
[ep 010] train_loss=0.6895  train_dec=0.5256  val_loss=0.6916  val_dec=0.5714
[ep 011] train_loss=0.6984  train_dec=0.5067  val_loss=0.6921  val_dec=0.5714
[ep 012] train_loss=0.6901  train_dec=0.5202  val_loss=0.6924  val_dec=0.5714
[ep 013] train_loss=0.6892  train_dec=0.5404

In [101]:
# model.load_state_dict(best_ckpt["state_dict"])
# model.to(device)
# model.eval()

# print(f"\n[best model restored from epoch {best_ckpt['epoch']}]")
# print(f" best val_dec={best_ckpt['dec_acc']:.4f}  best val_bub={best_ckpt['bub_acc']:.4f}\n")

# test_loss, test_dec = eval_choice_all(model, test_loader, device)

# print(f"[test-final] test_loss={test_loss:.4f}  test_dec={test_dec:.4f}")


[best model restored from epoch 199]
 best val_dec=0.6310  best val_bub=0.0000

[test-final] test_loss=0.7383  test_dec=0.5633


In [107]:
import math

def mean_std(xs):
    xs = list(xs)
    m = sum(xs) / max(len(xs), 1)
    v = sum((x - m) ** 2 for x in xs) / max(len(xs), 1)
    return m, math.sqrt(v)

RUNS = 10
EPOCHS = 200

all_test_loss = []
all_test_dec  = []
all_best_val  = []
all_best_ep   = []

for run in range(1, RUNS + 1):

    # -------- model / optim --------
    model = HyperbubbleGNN(
        vocab_size=6,
        embed_dim=32,
        gcn_hidden=128,
        edge_hidden=64,
        edge_feat_dim=4,
        use_lstm=False,
    ).to(device)

    total_params = sum(p.numel() for p in model.parameters())
    print(f"\n[run {run}/{RUNS}] params={total_params}")

    optim = opt.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-3)

    # -------- train / eval loop with best checkpoint (in-memory) --------
    best_ckpt = {"epoch": 0, "dec_acc": -1.0, "bub_acc": -1.0, "state_dict": None}

    def _is_better(bub_acc, dec_acc, best):
        if dec_acc > best["dec_acc"]:
            return True
        if dec_acc == best["dec_acc"] and bub_acc > best["bub_acc"]:
            return True
        return False

    for epoch in range(1, EPOCHS + 1):
        tr = train_one_epoch_choice_all(model, train_loader, optim, device)
        tr_loss = tr["loss"]
        tr_dec  = tr["dec_acc"]

        val_loss, val_dec = eval_choice_all(model, val_loader, device)

        print(f"[run {run} ep {epoch:03d}] "
              f"train_loss={tr_loss:.4f}  train_dec={tr_dec:.4f}  "
              f"val_loss={val_loss:.4f}  val_dec={val_dec:.4f}")

        # checkpoint logic
        dec_acc, bub_acc = val_dec, 0.0
        if _is_better(bub_acc, dec_acc, best_ckpt):
            best_ckpt["epoch"] = epoch
            best_ckpt["dec_acc"] = dec_acc
            best_ckpt["bub_acc"] = bub_acc
            best_ckpt["state_dict"] = {k: v.detach().cpu() for k, v in model.state_dict().items()}

    # restore best (no file I/O)
    model.load_state_dict(best_ckpt["state_dict"], strict=True)
    model.to(device)
    model.eval()

    print(f"\n[run {run}] best restored from epoch {best_ckpt['epoch']}: best_val_dec={best_ckpt['dec_acc']:.4f}")

    test_loss, test_dec = eval_choice_all(model, test_loader, device)
    print(f"[run {run} test] test_loss={test_loss:.4f}  test_dec={test_dec:.4f}")

    all_best_ep.append(best_ckpt["epoch"])
    all_best_val.append(best_ckpt["dec_acc"])
    all_test_loss.append(test_loss)
    all_test_dec.append(test_dec)

# -------- aggregate --------
m_loss, s_loss = mean_std(all_test_loss)
m_dec,  s_dec  = mean_std(all_test_dec)
m_val,  s_val  = mean_std(all_best_val)

print("\n===== 5-run summary =====")
print("Per-run best epochs: ", all_best_ep)
print("Per-run best val_dec:", [round(x, 4) for x in all_best_val])
print("Per-run test_loss:   ", [round(x, 4) for x in all_test_loss])
print("Per-run test_dec:    ", [round(x, 4) for x in all_test_dec])

print(f"\nMean best_val_dec = {m_val:.4f} ± {s_val:.4f}")
print(f"Mean test_loss    = {m_loss:.4f} ± {s_loss:.4f}")
print(f"Mean test_dec     = {m_dec:.4f} ± {s_dec:.4f}")


[run 1/10] params=76737
[run 1 ep 001] train_loss=0.6968  train_dec=0.5121  val_loss=0.6889  val_dec=0.5714
[run 1 ep 002] train_loss=0.6963  train_dec=0.5310  val_loss=0.6901  val_dec=0.5833
[run 1 ep 003] train_loss=0.6976  train_dec=0.5283  val_loss=0.6933  val_dec=0.5476
[run 1 ep 004] train_loss=0.6968  train_dec=0.5323  val_loss=0.6936  val_dec=0.5476
[run 1 ep 005] train_loss=0.6955  train_dec=0.5135  val_loss=0.6937  val_dec=0.5595
[run 1 ep 006] train_loss=0.6985  train_dec=0.5135  val_loss=0.6929  val_dec=0.5357
[run 1 ep 007] train_loss=0.6985  train_dec=0.5202  val_loss=0.6915  val_dec=0.5714
[run 1 ep 008] train_loss=0.6953  train_dec=0.5283  val_loss=0.6918  val_dec=0.5595
[run 1 ep 009] train_loss=0.6935  train_dec=0.5310  val_loss=0.6921  val_dec=0.5595
[run 1 ep 010] train_loss=0.6920  train_dec=0.5054  val_loss=0.6935  val_dec=0.5357
[run 1 ep 011] train_loss=0.6875  train_dec=0.5296  val_loss=0.6939  val_dec=0.5238
[run 1 ep 012] train_loss=0.6959  train_dec=0.5027 